# app

> FastHTML viewer: auth gate + month index + month view + authed downloads

In [ ]:
#| default_exp app

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, secrets
from pathlib import Path
from fasthtml.common import *
from starlette.responses import RedirectResponse, FileResponse
from starlette.exceptions import HTTPException
from reconcile_web.archive import list_months, month_counts, status_html, safe_file

In [ ]:
#| export
def create_app(
    archive_dir: str|Path|None = None,  # archive root (default: env ARCHIVE_DIR)
    password: str|None = None,          # shared password (default: env APP_PASSWORD)
    session_secret: str|None = None,    # session signing key (default: env SESSION_SECRET)
):
    "Build the FastHTML viewer app; raise RuntimeError on missing/invalid config"
    archive_dir = archive_dir or os.environ.get('ARCHIVE_DIR')
    app_password = password or os.environ.get('APP_PASSWORD')
    session_secret = session_secret or os.environ.get('SESSION_SECRET')
    missing = [n for n, v in (('ARCHIVE_DIR', archive_dir), ('APP_PASSWORD', app_password),
                              ('SESSION_SECRET', session_secret)) if not v]
    if missing: raise RuntimeError(f"missing config: {', '.join(missing)}")
    if not Path(archive_dir).is_dir(): raise RuntimeError(f"ARCHIVE_DIR is not a directory: {archive_dir}")

    def auth_before(req, session):
        if not session.get('auth'): return RedirectResponse('/login', status_code=303)

    def _check_month(month):
        if month not in list_months(archive_dir): raise HTTPException(404)

    def statement_links(m):
        return P(A('statement.pdf', href=f'/m/{m}/statement.pdf'), ' · ',
                 A('statement.csv', href=f'/m/{m}/statement.csv'))

    # skip list covers only /login (every other route requires auth); the app serves no local
    # static files (Pico CSS comes from the CDN), so fast_app's default static route is removed
    # right after creation below — its extension list (pdf/csv) would otherwise shadow the
    # dotted file routes (.pdf/.csv) defined further down
    app, rt = fast_app(before=Beforeware(auth_before, skip=[r'/login']),
                       secret_key=session_secret, sess_https_only=True)
    app.router.routes = [r for r in app.router.routes if getattr(r, 'path', None) != '/{fname:path}.{ext:static}']

    def login_page(error=False):
        body = [Form(Input(name='password', type='password', required=True),
                     Button('Login'), method='post', action='/login')]
        if error: body.insert(0, P('Wrong password'))
        return Titled('Login', *body)

    def expand_btn(m, oob=False):
        return Button('▸', hx_get=f'/m/{m}/expand', hx_target=f'#detail-{m}',
                      hx_swap='outerHTML', id=f'btn-{m}',
                      hx_swap_oob='true' if oob else None)

    def collapse_btn(m, oob=False):
        return Button('▾', hx_get=f'/m/{m}/collapse', hx_target=f'#detail-{m}',
                      hx_swap='outerHTML', id=f'btn-{m}',
                      hx_swap_oob='true' if oob else None)

    def month_row(m):
        c = month_counts(archive_dir, m)
        missing = c.get('missing', 0)
        return Tr(
            Td(expand_btn(m), A(m, href=f'/m/{m}')),
            Td(c.get('collected', 0)),
            Td(c.get('not-needed', 0)),
            Td(missing),
            cls='has-missing' if missing else None)

    def months_table():
        return Table(
            Thead(Tr(Th('month'), Th('collected'), Th('not-needed'), Th('missing'))),
            Tbody(*[el for m in reversed(list_months(archive_dir))
                    for el in (month_row(m), Tr(id=f'detail-{m}'))]))

    @rt('/login', methods=['GET'])
    def login_form(): return login_page()

    @rt('/login', methods=['POST'])
    def login_submit(password: str, session):
        if password and secrets.compare_digest(password.encode(), app_password.encode()):
            session['auth'] = True
            return RedirectResponse('/', status_code=303)
        return login_page(error=True)

    @rt('/logout', methods=['GET'])
    def logout(session):
        session.pop('auth', None)
        return RedirectResponse('/login', status_code=303)

    @rt('/', methods=['GET'])
    def index():
        return Titled('reconcile-archive',
                      Style('.has-missing td {color: var(--pico-del-color, #c62828)}\n'
                            'tbody button {width: auto; display: inline-block; padding: 0 .5em; margin-right: .5em}\n'
                            'tr[id^="detail-"] p:first-child {margin: 0; text-align: right; font-size: .85em}'),
                      months_table(),
                      P(A('Logout', href='/logout')))

    @rt('/m/{month}', methods=['GET'])
    def month_view(month: str):
        _check_month(month)
        return Titled(month,
            statement_links(month),
            NotStr(status_html(archive_dir, month)),
            P(A('← months', href='/')))

    def _file(month, kind, name=None):
        try: return FileResponse(safe_file(archive_dir, month, kind, name))
        except FileNotFoundError: raise HTTPException(404)

    @rt('/m/{month}/statement.pdf', methods=['GET'])
    def statement_pdf(month: str): return _file(month, 'statement_pdf')

    @rt('/m/{month}/statement.csv', methods=['GET'])
    def statement_csv(month: str): return _file(month, 'statement_csv')

    @rt('/m/{month}/receipt/{name}', methods=['GET'])
    def receipt(month: str, name: str): return _file(month, 'receipt', name)

    @rt('/m/{month}/expand', methods=['GET'])
    def expand(month: str):
        _check_month(month)
        return (Tr(Td(statement_links(month), NotStr(status_html(archive_dir, month)), colspan=4), id=f'detail-{month}'), collapse_btn(month, oob=True))

    @rt('/m/{month}/collapse', methods=['GET'])
    def collapse(month: str):
        _check_month(month)
        return Tr(id=f'detail-{month}'), expand_btn(month, oob=True)

    return app

In [ ]:
# test fixture: a throwaway archive matching the status.md contract in the spec
import tempfile
from pathlib import Path

STATUS_MD = """# {m} — receipt status

| date | title | amount | status | receipt_file | note |
|---|---|---|---|---|---|
| {m}-29 | OPENAI *CHATGPT SUBSCR | -110.46 | collected | [{m}-29_openai.pdf](receipts/{m}-29_openai.pdf) | src=x.pdf |
| {m}-06 | NORDEA | -20.46 | not-needed |  |  |

**missing (retrievable from Gmail/portal/Nordea): 1**
"""

def make_archive(months=('2025-07', '2025-08')):
    root = Path(tempfile.mkdtemp())
    (root/'README.md').write_text('readme')
    (root/'notamonth').mkdir()
    for m in months:
        d = root/m
        (d/'receipts').mkdir(parents=True)
        (d/'statement.pdf').write_bytes(b'%PDF-1.4 fake')
        (d/'statement.csv').write_text('date;amount\n')
        (d/'status.md').write_text(STATUS_MD.format(m=m))
        (d/'receipts'/f'{m}-29_openai.pdf').write_bytes(b'%PDF-1.4 fake receipt')
    return root

In [ ]:
from starlette.testclient import TestClient

root = make_archive()
app = create_app(archive_dir=root, password='pw', session_secret='s3cret')
client = TestClient(app, base_url='https://testserver')  # https: Secure cookie must round-trip

# unauthenticated -> redirected to /login
r = client.get('/', follow_redirects=False)
assert r.status_code == 303 and r.headers['location'] == '/login'
# static route removed; static-looking paths are not served (404, not accessible)
assert client.get('/static/x.pdf', follow_redirects=False).status_code == 404
# login page reachable without auth
assert client.get('/login').status_code == 200
# wrong password -> error notice, still no access
r = client.post('/login', data={'password': 'wrong'})
assert 'Wrong password' in r.text
assert client.get('/', follow_redirects=False).status_code == 303
# correct password -> index with months and counts
r = client.post('/login', data={'password': 'pw'}, follow_redirects=False)
assert r.status_code == 303
r = client.get('/')
assert r.status_code == 200 and '2025-07' in r.text and '2025-08' in r.text
assert '<th>collected</th>' in r.text and '<th>not-needed</th>' in r.text and '<th>missing</th>' in r.text and 'class="has-missing"' in r.text, r.text
assert r.text.index('2025-08') < r.text.index('2025-07')   # newest month first
r = client.get('/')   # ログイン済み client で
assert r.text.count('id="detail-') == 2      # 月数ぶんの placeholder 行
assert r.text.count('id="btn-') == 2         # 月数ぶんの ▸ ボタン
# logout ends the session
client.get('/logout', follow_redirects=False)
assert client.get('/', follow_redirects=False).status_code == 303
# non-ASCII password must not break login (compare_digest works on bytes)
app2 = create_app(archive_dir=root, password='pässwörd', session_secret='s3cret')
c2 = TestClient(app2, base_url='https://testserver')
assert c2.post('/login', data={'password': 'pässwörd'}, follow_redirects=False).status_code == 303
assert 'Wrong password' in c2.post('/login', data={'password': 'wröng'}).text
# missing config fails fast
try:
    create_app(archive_dir=root, password='pw'); assert False, 'should have raised'
except RuntimeError as e:
    assert 'SESSION_SECRET' in str(e)

In [ ]:
# missing=0 month gets no highlight
root = make_archive()
p = root/'2025-08'/'status.md'
p.write_text(p.read_text().replace(': 1', ': 0'))
app = create_app(archive_dir=root, password='pw', session_secret='s3cret')
client = TestClient(app, base_url='https://testserver')
client.post('/login', data={'password': 'pw'})
r = client.get('/')
assert r.text.count('class="has-missing"') == 1, r.text   # 2025-07 only

In [ ]:
root = make_archive()
app = create_app(archive_dir=root, password='pw', session_secret='s3cret')
client = TestClient(app, base_url='https://testserver')
client.post('/login', data={'password': 'pw'})

r = client.get('/m/2025-07')
assert r.status_code == 200
assert '/m/2025-07/receipt/2025-07-29_openai.pdf' in r.text   # rewritten receipt link
assert '/m/2025-07/statement.pdf' in r.text and '/m/2025-07/statement.csv' in r.text
assert client.get('/m/1999-01').status_code == 404            # unknown month

In [ ]:
root = make_archive()
app = create_app(archive_dir=root, password='pw', session_secret='s3cret')
client = TestClient(app, base_url='https://testserver')
client.post('/login', data={'password': 'pw'})

assert client.get('/m/2025-07/statement.pdf').content.startswith(b'%PDF')
assert client.get('/m/2025-07/statement.csv').status_code == 200
assert client.get('/m/2025-07/receipt/2025-07-29_openai.pdf').content.startswith(b'%PDF')
assert client.get('/m/2025-07/receipt/nope.pdf').status_code == 404
# downloads never bypass auth
anon = TestClient(app, base_url='https://testserver')
assert anon.get('/m/2025-07/receipt/2025-07-29_openai.pdf', follow_redirects=False).status_code == 303

In [ ]:
# expand partial: auth, guard, fragment
root = make_archive()
app = create_app(archive_dir=root, password='pw', session_secret='s3cret')
client = TestClient(app, base_url='https://testserver')
client.post('/login', data={'password': 'pw'})
anon = TestClient(app, base_url='https://testserver')
assert anon.get('/m/2025-07/expand', follow_redirects=False).status_code == 303   # auth
assert client.get('/m/not-a-month/expand').status_code == 404                     # guard
assert client.get('/m/1999-01/expand').status_code == 404
r = client.get('/m/2025-07/expand', headers={'HX-Request': 'true'})
assert 'id="detail-2025-07"' in r.text and '/m/2025-07/receipt/' in r.text
assert '<html' not in r.text                                                      # fragment, not a page
assert 'hx-swap-oob' in r.text
assert '/m/2025-07/statement.pdf' in r.text and '/m/2025-07/statement.csv' in r.text

In [ ]:
# collapse partial: auth, guard, empty placeholder + OOB ▸
root = make_archive()
app = create_app(archive_dir=root, password='pw', session_secret='s3cret')
client = TestClient(app, base_url='https://testserver')
client.post('/login', data={'password': 'pw'})

anon = TestClient(app, base_url='https://testserver')
assert anon.get('/m/2025-07/collapse', follow_redirects=False).status_code == 303
assert client.get('/m/not-a-month/collapse').status_code == 404
r = client.get('/m/2025-07/collapse', headers={'HX-Request': 'true'})
assert 'id="detail-2025-07"' in r.text and '/m/2025-07/receipt/' not in r.text   # 空の placeholder
assert 'id="btn-2025-07"' in r.text and 'hx-swap-oob' in r.text                  # OOB の ▸

In [ ]:
#| export
def serve(
    host: str = '0.0.0.0',  # bind address
    port: int = 8000,       # port
):
    "Run the viewer under uvicorn; config comes from env"
    import uvicorn
    uvicorn.run(create_app(), host=host, port=port)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| eval: false
root = make_archive()
app = create_app(archive_dir=root, password='pw', session_secret='s3cret')
client = TestClient(app, base_url='https://testserver')
client.post('/login', data={'password': 'pw'})

from IPython.display import HTML
HTML(client.get('/').text)

month,collected,not-needed,missing
▸2025-08,1,1,1
▸2025-07,1,1,1


In [ ]:
#| eval: false
# browser smoke: sess_https_only を無効化した dev app を JupyUvi で起動
from starlette.middleware.sessions import SessionMiddleware
from fasthtml.jupyter import JupyUvi

root = make_archive()
app_dev = create_app(archive_dir=root, password='pw', session_secret='s3cret')
for mw in app_dev.user_middleware:                      # 実験用: Secure cookie 要件だけ外す
  if mw.cls is SessionMiddleware: mw.kwargs['https_only'] = False
server = JupyUvi(app_dev, port=8000)
# 終わったら: server.stop()

In [ ]:
#| eval: false
server.stop()